# Linajea × Biohub: kiểm tra đọc dữ liệu trên Kaggle

Gắn competition **Biohub – Cell Tracking During Development** vào notebook và bật **Internet** trước khi chạy. Notebook này clone fork `hathanhv/linajea`, đọc một cặp Zarr v3/GEFF và tạo một batch Gunpowder nhỏ. Không train hoặc ghi vào competition input.

In [ ]:
from pathlib import Path
import subprocess
import sys

DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
REPO_DIR = Path('/kaggle/working/linajea')
BRANCH = 'biohub-io'
SAMPLE = None  # ví dụ: '44b6_341df25f'; None chọn cặp đầu tiên

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Không thấy {DATA_DIR}. Kiểm tra Kaggle Input và sửa DATA_DIR nếu mount ở nơi khác.')
print('Train directory:', DATA_DIR)
print('Zarr stores:', len(list(DATA_DIR.glob('*.zarr'))))
print('GEFF stores:', len(list(DATA_DIR.glob('*.geff'))))

In [ ]:
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO_DIR)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Linajea commit:', commit)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(REPO_DIR / 'requirements-biohub-io.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR),
                '--no-deps'], check=True)

In [ ]:
command = [sys.executable, str(REPO_DIR / 'run_scripts' / 'check_biohub_io.py'),
           '--data-dir', str(DATA_DIR)]
if SAMPLE is not None:
    command.extend(['--sample', SAMPLE])
subprocess.run(command, cwd=REPO_DIR, check=True)

Kết quả đạt khi cell cuối in `Biohub I/O smoke test passed`, cùng shape/dtype, voxel size, số node/cạnh và vị trí centroid trong batch. Nếu thiếu dependency hoặc dữ liệu có định dạng khác, traceback sẽ chỉ rõ bước lỗi.